# Data Preparation for Zim-my Fine-tuning

This notebook prepares the Zimbabwean and Shona datasets for fine-tuning Qwen2.5-3B-Instruct.

## Steps:
1. Load and explore downloaded datasets
2. Clean and format data into instruction-following format
3. Mix Shona and English data (70/30 ratio)
4. Create train/val/test splits
5. Save processed datasets

In [ ]:
import os
import json
from datasets import load_from_disk, Dataset
import pandas as pd
from tqdm import tqdm
import random

# Set random seed for reproducibility
random.seed(42)

## 1. Load and Explore Datasets

In [ ]:
# Load all datasets
datasets_dict = {}

dataset_paths = [
    ('ruzivo_shona_rag', 'data/raw/ruzivo-shona-rag'),
    ('alpaca_shona', 'data/raw/alpaca_shona_taco'),
    ('zimbabwe_agriculture', 'data/raw/zimbabwe_agriculture'),
    ('zimbabwe_history', 'data/raw/zimbabwe_history'),
    ('code_shona', 'data/raw/code_shona_10k'),
    ('african_math', 'data/raw/african_math'),
]

for name, path in dataset_paths:
    if os.path.exists(path):
        try:
            ds = load_from_disk(path)
            datasets_dict[name] = ds
            print(f"✓ Loaded {name}: {len(ds)} records")
            print(f"  Columns: {ds.column_names}")
            print(f"  Sample: {ds[0]}")
            print()
        except Exception as e:
            print(f"✗ Error loading {name}: {e}")
    else:
        print(f"✗ {name} not found at {path}")

## 2. Format Datasets for Instruction Tuning

We'll convert each dataset into a standard instruction-following format:
```json
{
  "instruction": "...",
  "input": "...",
  "output": "..."
}
```

In [ ]:
def format_alpaca_shona(dataset):
    """Format Alpaca Shona dataset"""
    formatted = []
    for item in tqdm(dataset, desc="Formatting Alpaca Shona"):
        formatted.append({
            'instruction': item.get('instruction', ''),
            'input': item.get('input', ''),
            'output': item.get('output', ''),
            'language': 'shona'
        })
    return formatted

def format_ruzivo_rag(dataset):
    """Format Ruzivo Shona RAG dataset into Q&A pairs"""
    formatted = []
    for item in tqdm(dataset, desc="Formatting Ruzivo RAG"):
        # Create Q&A from content
        if 'text' in item:
            text = item['text']
            # Create a simple Q&A format
            formatted.append({
                'instruction': 'Pindura mubvunzo uyu muShona:',
                'input': text[:500] if len(text) > 500 else text,
                'output': text,
                'language': 'shona'
            })
    return formatted

def format_agriculture(dataset):
    """Format Zimbabwe agriculture dataset"""
    formatted = []
    for item in tqdm(dataset, desc="Formatting Agriculture"):
        # Adapt based on actual structure
        instruction = item.get('question', item.get('instruction', 'Answer this agriculture question:'))
        context = item.get('context', item.get('input', ''))
        answer = item.get('answer', item.get('output', ''))
        
        formatted.append({
            'instruction': instruction,
            'input': context,
            'output': answer,
            'language': 'english'
        })
    return formatted

def format_history(dataset):
    """Format Zimbabwe history dataset"""
    formatted = []
    for item in tqdm(dataset, desc="Formatting History"):
        instruction = item.get('question', 'Tell me about Zimbabwe history:')
        context = item.get('context', item.get('text', ''))
        answer = item.get('answer', context)
        
        formatted.append({
            'instruction': instruction,
            'input': '',
            'output': answer,
            'language': 'english'
        })
    return formatted

def format_code_shona(dataset):
    """Format Shona code dataset"""
    formatted = []
    for item in tqdm(dataset, desc="Formatting Code Shona"):
        instruction = item.get('instruction', 'Write code:')
        input_text = item.get('input', '')
        output = item.get('output', item.get('code', ''))
        
        formatted.append({
            'instruction': instruction,
            'input': input_text,
            'output': output,
            'language': 'shona'
        })
    return formatted

def format_math(dataset):
    """Format African math dataset"""
    formatted = []
    for item in tqdm(dataset, desc="Formatting Math"):
        # Filter for Shona if available
        lang = item.get('language', 'english')
        instruction = item.get('question', item.get('instruction', 'Solve this math problem:'))
        solution = item.get('answer', item.get('solution', item.get('output', '')))
        
        formatted.append({
            'instruction': instruction,
            'input': '',
            'output': solution,
            'language': lang if lang in ['shona', 'english'] else 'english'
        })
    return formatted

In [ ]:
# Format all datasets
all_formatted = []

if 'alpaca_shona' in datasets_dict:
    all_formatted.extend(format_alpaca_shona(datasets_dict['alpaca_shona']))

if 'ruzivo_shona_rag' in datasets_dict:
    # Sample 50K from RAG corpus to keep training manageable
    rag_sample = datasets_dict['ruzivo_shona_rag'].select(range(min(50000, len(datasets_dict['ruzivo_shona_rag']))))
    all_formatted.extend(format_ruzivo_rag(rag_sample))

if 'zimbabwe_agriculture' in datasets_dict:
    all_formatted.extend(format_agriculture(datasets_dict['zimbabwe_agriculture']))

if 'zimbabwe_history' in datasets_dict:
    all_formatted.extend(format_history(datasets_dict['zimbabwe_history']))

if 'code_shona' in datasets_dict:
    all_formatted.extend(format_code_shona(datasets_dict['code_shona']))

if 'african_math' in datasets_dict:
    all_formatted.extend(format_math(datasets_dict['african_math']))

print(f"\n✓ Total formatted examples: {len(all_formatted)}")

## 3. Mix Shona and English Data

In [ ]:
# Separate by language
shona_data = [item for item in all_formatted if item['language'] == 'shona']
english_data = [item for item in all_formatted if item['language'] == 'english']

print(f"Shona examples: {len(shona_data)}")
print(f"English examples: {len(english_data)}")

# Target 70% Shona, 30% English
total_target = min(len(all_formatted), 100000)  # Cap at 100K for training
shona_target = int(total_target * 0.7)
english_target = total_target - shona_target

# Sample if needed
shona_sampled = random.sample(shona_data, min(shona_target, len(shona_data)))
english_sampled = random.sample(english_data, min(english_target, len(english_data)))

# Combine
mixed_data = shona_sampled + english_sampled
random.shuffle(mixed_data)

print(f"\n✓ Mixed dataset: {len(mixed_data)} examples")
print(f"  Shona: {len(shona_sampled)} ({len(shona_sampled)/len(mixed_data)*100:.1f}%)")
print(f"  English: {len(english_sampled)} ({len(english_sampled)/len(mixed_data)*100:.1f}%)")

## 4. Create Train/Val/Test Splits

In [ ]:
# Split: 90% train, 5% val, 5% test
n = len(mixed_data)
train_end = int(n * 0.90)
val_end = int(n * 0.95)

train_data = mixed_data[:train_end]
val_data = mixed_data[train_end:val_end]
test_data = mixed_data[val_end:]

print(f"Train: {len(train_data)} examples")
print(f"Validation: {len(val_data)} examples")
print(f"Test: {len(test_data)} examples")

## 5. Save Processed Datasets

In [ ]:
# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

# Save to disk
os.makedirs('data/processed', exist_ok=True)

train_dataset.save_to_disk('data/processed/train')
val_dataset.save_to_disk('data/processed/val')
test_dataset.save_to_disk('data/processed/test')

print("✓ Saved processed datasets to data/processed/")

# Also save as JSON for inspection
with open('data/processed/train_sample.json', 'w', encoding='utf-8') as f:
    json.dump(train_data[:100], f, ensure_ascii=False, indent=2)

print("✓ Saved sample to data/processed/train_sample.json")

## 6. Prepare RAG Corpus

In [ ]:
# Extract RAG corpus for retrieval
if 'ruzivo_shona_rag' in datasets_dict:
    rag_corpus = []
    for item in tqdm(datasets_dict['ruzivo_shona_rag'], desc="Building RAG corpus"):
        if 'text' in item:
            rag_corpus.append({
                'text': item['text'],
                'metadata': item.get('metadata', {})
            })
    
    # Save RAG corpus
    with open('data/rag/corpus.json', 'w', encoding='utf-8') as f:
        json.dump(rag_corpus, f, ensure_ascii=False)
    
    print(f"✓ Saved RAG corpus: {len(rag_corpus)} documents")
else:
    print("✗ Ruzivo Shona RAG dataset not found")

## Summary

Data preparation complete! Next steps:

1. Review `data/processed/train_sample.json` to verify data quality
2. Open `notebooks/02_finetune.ipynb` to start fine-tuning
3. The fine-tuning notebook will load from `data/processed/train` and `data/processed/val`